Import relevant libraries:

In [2]:
import numpy as np, matplotlib.pyplot as plt, scipy as sci, gzip, time
from scipy import special
from pathlib import Path

Load files:

In [7]:

# Get the folder this script is located in
BASE_PATH = Path("MNIST Neural Net.ipynb").parent

# Point to the MNIST raw data folder
DATA_PATH = BASE_PATH / "data" / "MNIST" / "raw"


# ================= TRAIN IMAGES =================

with gzip.open(DATA_PATH / "train-images-idx3-ubyte.gz", "rb") as f:
	data = np.frombuffer(f.read(), dtype=np.uint8, offset=16)
	images = data.reshape(-1, 28, 28)
	train_flattened = images.reshape(-1, 784)


# ================= TRAIN LABELS =================

with gzip.open(DATA_PATH / "train-labels-idx1-ubyte.gz", "rb") as f:
	train_labels = np.frombuffer(f.read(), dtype=np.uint8, offset=8)


# ================= TEST IMAGES =================

with gzip.open(DATA_PATH / "t10k-images-idx3-ubyte.gz", "rb") as f:
	data = np.frombuffer(f.read(), dtype=np.uint8, offset=16)
	images = data.reshape(-1, 28, 28)
	test_flattened = images.reshape(-1, 784)


# ================= TEST LABELS =================

with gzip.open(DATA_PATH / "t10k-labels-idx1-ubyte.gz", "rb") as f:
	test_labels = np.frombuffer(f.read(), dtype=np.uint8, offset=8)


# Normalize images
train_flattened = train_flattened.astype(np.float32) / 255.0
test_flattened = test_flattened.astype(np.float32) / 255.0


print("Loaded MNIST!")
print("Training images:", train_flattened.shape)
print("Training labels:", train_labels.shape)
print("Testing images:", test_flattened.shape)
print("Testing labels:", test_labels.shape)

Loaded MNIST!
Training images: (60000, 784)
Training labels: (60000,)
Testing images: (10000, 784)
Testing labels: (10000,)


Define activation function and neural net classes:

In [ ]:
def sigmoid(x):
    #return 1 / (1 + np.exp(-x))
    return sci.special.expit(x).astype(np.float32)

class Layer:
    def __init__(self, input_size : int, output_size : int):
        self.input_size = input_size
        self.output_size = output_size
        
        # Initialise weights with random values
        scale = np.float32(np.sqrt(np.float32(1.0 / input_size)))
        self.weights = (np.random.randn(output_size, input_size)).astype(np.float32) * scale
        self.biases = np.zeros(output_size, dtype = np.float32)

    def get_activation(self, input: np.ndarray):
        if input.size != self.input_size:
            raise Exception(f"Error: Input size ({input.size}) does not match expected input size ({self.input_size})!")
        
        # Return both the sigmoided values and z values.
        z_values = np.matmul(self.weights, input) + self.biases
        return [sigmoid(z_values), z_values]

class Network:
    def __init__(self, input_layer_size: int, output_layer_size: int, hidden_layers: list[int]):
        # Create input layer
        self.input_layer = np.zeros(input_layer_size, dtype = np.float32)

        self.hidden_layers : list[Layer] = []
        prev_layer_input = input_layer_size
        
        # Create hidden layers
        for layer in hidden_layers:
            self.hidden_layers.append(Layer(prev_layer_input, layer))
            prev_layer_input = layer

        self.output_layer = Layer(prev_layer_input, output_layer_size)

    def forward_pass(self, input: np.ndarray):
        self.set_input_layer(input)
        activations : list = [self.input_layer]
        z_values : list = []
        
        # Hidden layers
        for layer in self.hidden_layers:
            current = layer.get_activation(activations[-1])
            activations.append(current[0])
            z_values.append(current[1])
        
        # Output layer
        current = self.output_layer.get_activation(activations[-1])
        activations.append(current[0])
        z_values.append(current[1])
        return [activations, z_values]

    def backpropagation(self, input : np.ndarray, label : int):
        activations, z_values = self.forward_pass(input)
        one_hot = np.zeros(10, dtype = np.float32)
        one_hot[label] = 1
        
        # ======== OUTPUT LAYER ========
        output = activations[-1]
        
        # How wrong each neuron is * sigmoid sensitivity
        delta : np.float32 = (one_hot - output) * output * (1 - output)
        
        # Last hidden layer
        previous_activation = activations[-2]
        
        weight_gradient : np.float32 = np.outer(delta, previous_activation)
        
        # Update output layer
        output_layer_weight_change = weight_gradient
        output_layer_bias_change = delta
        
        # ======== HIDDEN LAYERS ========
        
        next_delta = delta
        next_weights = self.output_layer.weights
        hidden_layer_weight_change = []
        hidden_layer_bias_change = []
        
        for i in range(len(self.hidden_layers) - 1, -1, -1):
            
            # Activation of this layer
            activation = activations[i + 1]
            
            # Propagate "blame" backwards
            delta = np.matmul(next_weights.T, next_delta)
            
            # Apply sigmoid sensitivity
            delta = delta * activation * (1 - activation)
            
            # Activation of previous layer
            previous_activation = activations[i]
            
            # Gradient of this layers weights
            weight_gradient = np.outer(delta, previous_activation)
            
            # Update this layer
            hidden_layer_weight_change.append(weight_gradient)
            hidden_layer_bias_change.append(delta)
            
            # Prep for next layer
            next_delta = delta
            next_weights = self.hidden_layers[i].weights
        hidden_layer_weight_change.reverse()
        hidden_layer_bias_change.reverse()
        return output_layer_weight_change, output_layer_bias_change, hidden_layer_weight_change, hidden_layer_bias_change

    def get_prediction(self, input : np.ndarray):
        output = self.forward_pass(input)[0][-1]
        return(np.argmax(output))

    def set_input_layer(self, arr : np.ndarray):
        if arr.size != self.input_layer.size:
            raise Exception(f"Error: Input size ({arr.size}) does not match expected input size ({self.input_layer.size})!")
        self.input_layer = arr

    def get_loss(self, output : np.ndarray, label : int):
        one_hot = np.zeros(10, dtype=int)
        one_hot[label] = 1
        
        return np.mean((output-one_hot) ** 2)
    
    def train(self, data : np.ndarray, labels : np.ndarray, batch_amount, learning_rate : float, random_permutations : bool = True, epochs : int = 1):
        learning_rate = np.float32(learning_rate)
        training_start_time = time.time()
        print(f"Started training with {epochs} epochs...")
        for epoch in range(epochs):
            epoch_start_time = time.time()
            
            # Randomize permutations
            if random_permutations:
                p = np.random.permutation(len(data))
                data = data[p]
                labels = labels[p]
            
            # Split data into batches
            batches = np.array_split(data, batch_amount)
            batch_labels = np.array_split(labels, batch_amount)
            
            for b in range(len(batches)):
                batch = batches[b]
                label = batch_labels[b]
                batch_size = len(batch)
                sum_output_w = np.zeros_like(self.output_layer.weights)
                sum_output_b = np.zeros_like(self.output_layer.biases)
                
                sum_hidden_w = [np.zeros_like(layer.weights) for layer in self.hidden_layers]
                sum_hidden_b = [np.zeros_like(layer.biases) for layer in self.hidden_layers]
                
                # Store gradients
                for i in range(batch_size):
                    ow, ob, hw, hb = self.backpropagation(batch[i], label[i])
                    sum_output_w += ow
                    sum_output_b += ob
                    
                    for layer, (w, b) in enumerate(zip(hw, hb)):
                        sum_hidden_w[layer] += w
                        sum_hidden_b[layer] += b
                
                # Learning / batch size for averaging
                scale = learning_rate / np.float32(batch_size)
                
                self.output_layer.weights += sum_output_w * scale
                self.output_layer.biases += sum_output_b * scale
                
                for layer in range(len(self.hidden_layers)):
                    self.hidden_layers[layer].weights += sum_hidden_w[layer] * scale
                    self.hidden_layers[layer].biases += sum_hidden_b[layer] * scale

            print(f"Epoch {epoch+1}/{epochs} completed in {round(time.time() - epoch_start_time, 2)}s")
        print(f"Training completed in {round(time.time() - training_start_time, 2)}s")
        
        
    def save(self, filename : str = "model.npz"):
        data = {}
        
        for i, layer in enumerate(self.hidden_layers):
            data[f"hidden_{i}_weights"] = layer.weights
            data[f"hidden_{i}_biases"] = layer.biases
        
        data["output_weights"] = self.output_layer.weights
        data["output_biases"] = self.output_layer.biases
        
        np.savez(filename, **data)
    
    def load(self, filename : str = "model.npz"):
        try:
            data = np.load(filename)
        except FileNotFoundError:
            print("Model file not found. Maybe you made a typo?")
            return
        
        for i, layer in enumerate(self.hidden_layers):
            layer.weights = data[f"hidden_{i}_weights"]
            layer.biases = data[f"hidden_{i}_biases"]
        
        self.output_layer.weights = data["output_weights"]
        self.output_layer.biases = data["output_biases"]

Training:


In [48]:
net = Network(784, 10, [64, 32])

net.train(train_flattened, train_labels, 1000, 1, epochs=5)




Started training with 5 epochs...
Epoch 1/5 completed in 46.46s
Epoch 2/5 completed in 17.39s


KeyboardInterrupt: 

Check accuracy:

In [45]:
amount_right = 0
amount = 0
loss_sum = 0
for i in range(len(test_flattened)):
    prediction = net.get_prediction(test_flattened[i])
    if prediction == test_labels[i]:
        amount_right += 1
    amount += 1
    loss_sum += net.get_loss(net.forward_pass(test_flattened[i])[0][-1], test_labels[i])

print(f"Average loss: {loss_sum/len(test_flattened)}, Accuracy: {amount_right/amount*100}%")

Average loss: 0.007026120293978798, Accuracy: 95.95%


In [28]:
net.save()

In [43]:
net.load()